# Part 3: Hyperparameter, Data-Strategy and Ablation Studies
**Course:** AI403 Deep Learning · Project: Wearable HAR (Motion Mindset)

All studies in this notebook are evaluated on the **validation subjects only** (subjects 1, 3, 5, 6). The official test set is never used here, so model selection does not leak test information.

**Protocol (same as notebooks 1 and 2):** subject-wise split, train-only Z-score, Adam (lr 1e-3, clipnorm 1.0), batch size 64, up to 40 epochs, `EarlyStopping(val_loss, patience=8)`, seeds 42 / 123 / 2026, reported as mean ± SD.

| Study | Question | Link |
|---|---|---|
| 1. Hyperparameters | How do hidden units and learning rate affect the Bi-LSTM? | tuning |
| 2. Data strategy | Does Z-score scaling or jitter augmentation help? | representation / augmentation |
| 3a. Channel ablation | Which sensor channels matter most? | **H3** |
| 3b. Component ablation | Do the CNN and the custom attention help the hybrid? | **H2** |

Outputs are written to `../results/study_*.csv`. Estimated runtime: 25–40 minutes (about 25 configurations × 3 seeds).

### Step 1: Setup, data and helper functions

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import keras
from keras import layers
from sklearn.metrics import accuracy_score, f1_score

ROOT = os.path.abspath("..")                 # notebook lives in <repo>/notebooks
sys.path.insert(0, os.path.join(ROOT, "src"))
from datasets.loader import get_har_data
from models.custom_layers import TemporalAttentionLayer

SEEDS = [42, 123, 2026]
OUT = os.path.join(ROOT, "results"); os.makedirs(OUT, exist_ok=True)

# Channel order of src/datasets/loader.py (SIGNAL_TYPES) -- used by name in the ablation
BODY_ACC, GYRO, TOTAL_ACC = [0, 1, 2], [3, 4, 5], [6, 7, 8]

_, _, _, (Xtr, ytr, Xv, yv, _X_test, _y_test), stats = get_har_data(
    os.path.join(ROOT, "data", "UCI HAR Dataset"))
print("train", Xtr.shape, "| val", Xv.shape, "(test set not used in this notebook)")

In [ ]:
def compile_(m, lr=1e-3):
    m.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def bilstm(shape, units=64, lr=1e-3, dropout=0.2):
    i = layers.Input(shape=shape)
    x = layers.Bidirectional(layers.LSTM(units, dropout=dropout))(i)
    x = layers.Dropout(0.3)(x)
    return compile_(keras.Model(i, layers.Dense(6, activation="softmax")(x)), lr)

def hybrid(shape, cnn=True, attention=True, lr=1e-3):
    i = layers.Input(shape=shape); x = i
    if cnn:
        x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x); x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.2))(x)
    x = TemporalAttentionLayer(attention_dim=64)(x) if attention else layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    return compile_(keras.Model(i, layers.Dense(6, activation="softmax")(x)), lr)

def run(name, build, Xtr, ytr, Xv, yv):
    """Train `build` over all seeds; return validation mean ± SD (test set untouched)."""
    accs, f1s, times = [], [], []
    for seed in SEEDS:
        keras.utils.set_random_seed(seed)
        m = build(Xtr.shape[1:])
        es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
        t0 = time.time()
        m.fit(Xtr, ytr, validation_data=(Xv, yv), epochs=40, batch_size=64, callbacks=[es], verbose=0)
        times.append(time.time() - t0)
        p = np.argmax(m.predict(Xv, verbose=0), axis=1)
        accs.append(accuracy_score(yv, p) * 100); f1s.append(f1_score(yv, p, average="macro") * 100)
    print(f"{name:55s} val acc {np.mean(accs):.2f} ± {np.std(accs):.2f}")
    return {"Experiment": name, "Parameters": m.count_params(),
            "Val Accuracy (%)": f"{np.mean(accs):.2f} ± {np.std(accs):.2f}",
            "Val Macro-F1 (%)": f"{np.mean(f1s):.2f} ± {np.std(f1s):.2f}",
            "Train Time (s)": round(float(np.mean(times)), 1)}

### Step 2: Hyperparameter study (Bi-LSTM): hidden units × learning rate

In [ ]:
rows = [run(f"Bi-LSTM units={u}, lr={lr:g}", lambda s, u=u, lr=lr: bilstm(s, u, lr), Xtr, ytr, Xv, yv)
        for u in (32, 64, 128) for lr in (1e-3, 3e-4)]
hp_df = pd.DataFrame(rows)
hp_df.to_csv(os.path.join(OUT, "study_hyperparameters.csv"), index=False)
hp_df

### Step 3: Data-strategy study (Bi-LSTM, 64 units): scaling and augmentation
Z-score (train statistics) vs. raw unscaled input vs. Z-score + Gaussian jitter augmentation (σ = 0.05, training set doubled).

In [ ]:
rng = np.random.default_rng(0)
Xtr_raw = Xtr * stats["std"] + stats["mean"]
Xv_raw  = Xv  * stats["std"] + stats["mean"]
Xtr_aug = np.concatenate([Xtr, Xtr + rng.normal(0, 0.05, Xtr.shape).astype(np.float32)])
ytr_aug = np.concatenate([ytr, ytr])

rows = [run("Z-score (baseline strategy)", bilstm, Xtr, ytr, Xv, yv),
        run("Raw unscaled", bilstm, Xtr_raw, ytr, Xv_raw, yv),
        run("Z-score + jitter augmentation (sigma=0.05)", bilstm, Xtr_aug, ytr_aug, Xv, yv)]
ds_df = pd.DataFrame(rows)
ds_df.to_csv(os.path.join(OUT, "study_data_strategy.csv"), index=False)
ds_df

### Step 4: Ablation study
**4a. Sensor channels (Bi-LSTM):** channels are selected *by name* using the loader's order (body acc 0–2, gyroscope 3–5, total acc 6–8). This tests **H3**.
**4b. Hybrid components:** remove the custom attention (replaced by global average pooling) or the 1D-CNN front-end. This tests **H2**.

In [ ]:
sel = {"Full (9 ch)":          list(range(9)),
       "Without gyroscope":    BODY_ACC + TOTAL_ACC,
       "Without total acc":    BODY_ACC + GYRO,
       "Without body acc":     GYRO + TOTAL_ACC}
rows = [run(f"Bi-LSTM | {n}", bilstm, Xtr[:, :, c], ytr, Xv[:, :, c], yv) for n, c in sel.items()]
rows += [run("Hybrid | full (CNN + BiLSTM + Attention)", hybrid, Xtr, ytr, Xv, yv),
         run("Hybrid | without attention (GAP)", lambda s: hybrid(s, attention=False), Xtr, ytr, Xv, yv),
         run("Hybrid | without CNN",             lambda s: hybrid(s, cnn=False),       Xtr, ytr, Xv, yv)]
ab_df = pd.DataFrame(rows)
ab_df.to_csv(os.path.join(OUT, "study_ablation.csv"), index=False)
ab_df

### Step 5: Notes for the report
* Everything above uses validation data only. Report these numbers in Tables 3 and 5 as **validation** results.
* Differences smaller than about one standard deviation should be described as "no clear difference".
* The final test numbers in `final_4models_comparison.csv` come from notebooks 1–2 with fixed configurations and are not affected by this notebook.